# Engenharia de Features

**Objetivo:** Construir features de domínio e validar o pipeline de pré-processamento.

---


## 0. Configuração

In [ ]:
import sys
sys.path.insert(0, "../src")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data.preprocessing import load_raw_data, clean_data, split_data
from features.build_features import (
    ChurnFeatureEngineer,
    build_preprocessor,
    build_full_pipeline,
    CATEGORICAL_FEATURES,
    NUMERICAL_FEATURES,
)
from sklearn.linear_model import LogisticRegression


## 1. Carregamento dos Dados Limpos

In [ ]:
df_bruto = load_raw_data("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df = clean_data(df_bruto)
X_treino, X_teste, y_treino, y_teste = split_data(df, test_size=0.2, random_state=42)
print(f"Treino: {X_treino.shape} | Teste: {X_teste.shape}")


## 2. Engenharia de Features de Domínio

In [ ]:
engenheiro = ChurnFeatureEngineer()
X_treino_eng = engenheiro.fit_transform(X_treino)

# Exibe as novas features
novas_features = ["n_servicos", "gasto_mensal_medio", "gasto_por_servico",
                  "tendencia_cobranca", "contrato_longo_prazo", "grupo_tenure"]
print("Novas features de domínio:")
X_treino_eng[novas_features].describe().round(2)


In [ ]:
# Validação: distribuição de n_servicos vs churn
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

X_treino_eng["Churn"] = y_treino.values
churn_por_servicos = X_treino_eng.groupby("n_servicos")["Churn"].mean() * 100
axes[0].bar(churn_por_servicos.index, churn_por_servicos.values, color="#F44336", alpha=0.8)
axes[0].set_xlabel("Número de Serviços Adicionais")
axes[0].set_ylabel("Taxa de Churn (%)")
axes[0].set_title("Taxa de Churn por Número de Serviços", fontweight="bold")

for churn, cor, label in [(0, "#2196F3", "Sem Churn"), (1, "#F44336", "Churn")]:
    subset = X_treino_eng[X_treino_eng["Churn"]==churn]["tendencia_cobranca"]
    subset.plot.kde(ax=axes[1], color=cor, label=label, linewidth=2)
axes[1].set_xlabel("Tendência de Cobrança (MonthlyCharges - GastoMedioMensal)")
axes[1].set_title("Distribuição da Tendência de Cobrança por Churn", fontweight="bold")
axes[1].legend()

X_treino_eng.drop(columns="Churn", inplace=True)
plt.tight_layout()
plt.savefig("../reports/figures/06_features_dominio.png", dpi=120, bbox_inches="tight")
plt.show()


## 3. Validação do Pipeline de Pré-processamento

In [ ]:
preprocessador = build_preprocessor()
X_treino_processado = preprocessador.fit_transform(X_treino_eng)
nomes_features = preprocessador.get_feature_names_out()

print(f"Shape de saída: {X_treino_processado.shape}")
print(f"Número de features após codificação: {len(nomes_features)}")
print()
print("Exemplos de nomes de features (primeiros 10):")
for nome in nomes_features[:10]:
    print(f"  {nome}")


In [ ]:
# Verificar NaN/Inf na saída pré-processada
tem_nan = np.isnan(X_treino_processado).any()
tem_inf = np.isinf(X_treino_processado).any()
print(f"Valores NaN: {tem_nan}")
print(f"Valores Inf: {tem_inf}")
assert not tem_nan, "NaN encontrado na saída pré-processada!"
assert not tem_inf, "Inf encontrado na saída pré-processada!"
print("Saída pré-processada está limpa.")


## 4. Modelo Baseline (Regressão Logística)

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

# Verificação rápida com RL antes do notebook de modelagem completo
pipeline_rl = build_full_pipeline(LogisticRegression(C=1.0, max_iter=1000, random_state=42))
pipeline_rl.fit(X_treino, y_treino)

y_prob_rl = pipeline_rl.predict_proba(X_teste)[:, 1]
print(f"Regressão Logística ROC-AUC (baseline): {roc_auc_score(y_teste, y_prob_rl):.4f}")
print()
print(classification_report(y_teste, (y_prob_rl >= 0.5).astype(int),
                              target_names=["Sem Churn", "Churn"]))


In [ ]:
# Importância de features via coeficientes da RL
modelo_rl = pipeline_rl.named_steps["classificador"]
preprocessador_ajustado = pipeline_rl.named_steps["preprocessador"]
nomes_feat = preprocessador_ajustado.get_feature_names_out()

df_coef = pd.DataFrame({
    "feature": nomes_feat,
    "coeficiente": modelo_rl.coef_[0],
}).sort_values("coeficiente", key=abs, ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
cores = ["#F44336" if c > 0 else "#2196F3" for c in df_coef["coeficiente"]]
ax.barh(df_coef["feature"], df_coef["coeficiente"], color=cores)
ax.set_title("Top 15 Coeficientes da Regressão Logística (Fatores de Churn)", fontweight="bold")
ax.set_xlabel("Coeficiente (positivo = aumenta risco de churn)")
ax.axvline(0, color="black", lw=0.8)
plt.tight_layout()
plt.savefig("../reports/figures/07_coeficientes_rl.png", dpi=120, bbox_inches="tight")
plt.show()


## 5. Resumo

- 6 features de domínio criadas com lógica de negócio clara
- Pipeline de pré-processamento gera matrizes limpas, sem NaN
- RL baseline atinge ~0.84 ROC-AUC — bom ponto de partida
- Features chave: tipo de contrato, tenure, mensalidade, suporte técnico

Próximo: Modelagem Completa com XGBoost + SHAP
